# Exercise 5 — Bivariate Spectral Analysis: Coherence

**Neural Data Science (WP7)** · LMU Biology

---

### Overview

**Coherence** quantifies the degree of phase-consistent coupling between two
neural signals as a function of frequency.  Unlike time-domain correlation
(which integrates over all frequencies), coherence tells you *at which
oscillation frequency* two signals communicate.

In this exercise you will:
1. Load hippocampal LFP data and select two channels
2. Compute the cross-spectrum and coherence
3. Build a null distribution to assess significance
4. Interpret frequency-specific coupling at theta and gamma

### Learning objectives

- Understand the relationship between cross-spectrum and coherence
- Compare DPSS multitaper vs Welch-based coherence estimates
- Build a null distribution via time-shifting to assess significance
- Interpret coherence results in terms of hippocampal network coupling

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.io
import scipy.signal
import sys
import os

# Add the course library to the path
sys.path.insert(0, '../../lib')
from wp7_helpers import coherence, cross_spectrum, psd_multitaper

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

### Load the data

Same 16-channel hippocampal LFP as Ex7 and Ex8, sampled at **1250 Hz**.
We select two channels at different positions on the shank.

> **Note:** The SS25 Exercise 10 notebook uses a 96-channel dataset (`data96.mat`)
> for a richer analysis. The 96-ch data is available in
> `sourcedata/moodle/ss25-public/.../Exercise_10/data96.mat` for interested students.
> Here we use the 16-ch shank data for simplicity and continuity with Ex7/Ex8.

In [ ]:
data_path = '../../data/spectral/ws_data_1shank.mat'
# Data is at ../../data/spectral/ relative to this exercise directory

if not os.path.exists(data_path):
    raise FileNotFoundError(
        f"Data not found at {data_path!r}. Adjust data_path to point to ws_data_1shank.mat"
    )

mat = scipy.io.loadmat(data_path)
lfps = mat['lfps']
fs = 1250  # Hz

n_samples, n_channels = lfps.shape
print(f'LFP shape: {lfps.shape} — {n_channels} channels, {n_samples/fs:.1f} s at {fs} Hz')

---
## Section 1 — Exploring Two LFP Channels

Select two channels (e.g., channels 0 and 8 — near and far on the shank)
and plot them together to see how similar (or different) they look.

> **Think about it:** Just from looking at the two traces, can you tell
> whether they oscillate at the same frequency?  Could they be correlated
> overall but only coherent at specific frequencies?

In [ ]:
# ──────────────────────────────────────────────────────
# YOUR TURN: Select and visualize two channels
# ──────────────────────────────────────────────────────
# 1. Choose two channel indices (e.g. ch_x=0, ch_y=8)
# 2. Extract the full signals as x and y
# 3. Plot the first 5 seconds of both in a 2-row figure
# ──────────────────────────────────────────────────────

ch_x = ...  # TODO: pick first channel
ch_y = ...  # TODO: pick second channel

raise NotImplementedError("Select channels and plot their time series")

<details>
<summary><b>Hint 1 — Conceptual</b></summary>

Pick two channels that are separated on the electrode shank (e.g., ch 0
and ch 8).  Close channels tend to be very similar; distant ones show
more interesting coherence structure.
</details>

<details>
<summary><b>Hint 2 — API</b></summary>

```python
x = lfps[:, ch_x]
y = lfps[:, ch_y]
```

Use `plt.subplots(2, 1, sharex=True)` for a stacked plot.
</details>

<details>
<summary><b>Hint 3 — Partial code</b></summary>

```python
ch_x, ch_y = 0, 8
x = lfps[:, ch_x]
y = lfps[:, ch_y]

seg = slice(0, int(5 * fs))
t = np.arange(seg.stop) / fs

fig, axes = plt.subplots(2, 1, figsize=(12, 4), sharex=True)
axes[0].plot(t, x[seg], linewidth=0.5)
axes[0].set_ylabel(f'Ch {ch_x}')
axes[0].set_title(f'LFP — Channels {ch_x} and {ch_y}')
axes[1].plot(t, y[seg], linewidth=0.5, color='C1')
axes[1].set_ylabel(f'Ch {ch_y}')
axes[1].set_xlabel('Time (s)')
plt.tight_layout()
plt.show()
```
</details>

In [ ]:
# Checkpoint 1
assert len(x) == n_samples and len(y) == n_samples, "x and y must be full-length signals"
assert ch_x != ch_y, "Pick two different channels"
print(f"Checkpoint 1 passed — channels {ch_x} and {ch_y}, {n_samples:,} samples each")

---
## Section 2 — Cross-Spectrum and Coherence

The **cross-spectral density (CSD)** captures the frequency-resolved
phase relationship between two signals.  **Magnitude-squared coherence**
normalizes by the individual power spectra:

$$C_{xy}(f) = \frac{|P_{xy}(f)|^2}{P_{xx}(f) \cdot P_{yy}(f)} \in [0, 1]$$

Compare multitaper coherence (`wp7_helpers.coherence`) with Welch-based
coherence (`scipy.signal.coherence`) to see how the choice of method matters.

> **Think about it:** Why is coherence bounded between 0 and 1?
> What does a coherence of 0.5 mean practically?

In [ ]:
# ──────────────────────────────────────────────────────
# YOUR TURN: Compute cross-spectrum and coherence
# ──────────────────────────────────────────────────────
# 1. Choose nperseg = int(2.0 * fs) for 2-second segments
# 2. Compute multitaper CSD: cross_spectrum(x, y, fs, nperseg, nw=3)
# 3. Compute multitaper coherence: coherence(x, y, fs, nperseg, nw=3)
# 4. Compute Welch coherence: scipy.signal.coherence(x, y, fs=fs, nperseg=nperseg)
# 5. Plot CSD magnitude and both coherence estimates
# ──────────────────────────────────────────────────────

nperseg = int(2.0 * fs)

raise NotImplementedError("Compute and compare cross-spectrum and coherence")

<details>
<summary><b>Hint 1 — Conceptual</b></summary>

`cross_spectrum(x, y, ...)` returns `(freqs, Pxy)` where `Pxy` is complex.
Its magnitude is the cross-spectral power; its phase gives the average lag.
`coherence(x, y, ...)` returns `(freqs, Cxy)` where `Cxy` is real-valued
and bounded [0, 1].
</details>

<details>
<summary><b>Hint 2 — API</b></summary>

```python
freqs_csd, Pxy = cross_spectrum(x, y, fs=fs, nperseg=nperseg, nw=3)
freqs_coh, Cxy = coherence(x, y, fs=fs, nperseg=nperseg, nw=3)
freqs_welch_coh, Cxy_welch = scipy.signal.coherence(x, y, fs=fs, nperseg=nperseg)
```
</details>

<details>
<summary><b>Hint 3 — Partial code</b></summary>

```python
freqs_csd, Pxy = cross_spectrum(x, y, fs=fs, nperseg=nperseg, nw=3)
freqs_coh, Cxy = coherence(x, y, fs=fs, nperseg=nperseg, nw=3)
freqs_welch_coh, Cxy_welch = scipy.signal.coherence(x, y, fs=fs, nperseg=nperseg)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
ax1.semilogy(freqs_csd, np.abs(Pxy), linewidth=0.8)
ax1.set_ylabel('|CSD|')
ax1.set_title(f'Cross-Spectrum and Coherence — Ch {ch_x} vs Ch {ch_y}')

ax2.plot(freqs_coh, Cxy, linewidth=0.8, label='Multitaper (NW=3)')
ax2.plot(freqs_welch_coh, Cxy_welch, linewidth=0.8, alpha=0.7, label='Welch')
ax2.set_xlabel('Frequency (Hz)')
ax2.set_ylabel('Coherence')
ax2.set_ylim(0, 1)
ax2.legend()

for ax in (ax1, ax2):
    ax.set_xlim(0, 200)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
```
</details>

In [ ]:
# Checkpoint 2
assert np.all(Cxy >= 0) and np.all(Cxy <= 1), (
    f"Coherence must be in [0, 1], got [{Cxy.min():.4f}, {Cxy.max():.4f}]"
)
print(f"Checkpoint 2 passed — coherence range: [{Cxy.min():.4f}, {Cxy.max():.4f}]")

---
## Section 3 — Null Distribution via Time-Shifting

Coherence is **biased high** with short segments — even independent signals
show non-zero coherence.  Without a null, you might over-interpret small peaks.

We build a null distribution by circularly shifting one signal by a random
lag and recomputing coherence.  This destroys the phase relationship while
preserving the spectral content of each signal.

> **Think about it:** Why use a *circular shift* rather than generating
> a new random signal?  What property of the data does this preserve?

In [ ]:
# ──────────────────────────────────────────────────────
# YOUR TURN: Build the null distribution
# ──────────────────────────────────────────────────────
# For each of n_null iterations:
#   1. Circularly shift y by a random lag (at least 1 second)
#      Use np.roll(y, lag)
#   2. Recompute coherence between x and y_shifted
#   3. Store the coherence spectrum
# Then compute the 95th percentile at each frequency.
# ──────────────────────────────────────────────────────

n_null = 200
rng = np.random.default_rng(42)

raise NotImplementedError("Build the null distribution via time-shifting")

<details>
<summary><b>Hint 1 — Why do we need a null at all?</b></summary>

Coherence is estimated from finite data.  Even two completely independent
signals will show non-zero coherence simply due to statistical fluctuations.
The shorter your segments, the higher this "floor" of spurious coherence.
A null distribution tells you where that floor is, so you only interpret
coherence peaks that exceed it.
</details>

<details>
<summary><b>Hint 2 — How to build a null via time-shuffle</b></summary>

Circularly shift `y` by a random lag using `np.roll(y, lag)`.  The lag
should be large enough to break the phase relationship (at least 1 second).
For each shifted version, recompute coherence and store the result.
Then take the 95th percentile across all null coherences at each frequency.

```python
null_coherences = np.zeros((n_null, len(freqs_coh)))
for i in range(n_null):
    lag = rng.integers(int(1 * fs), len(y) - 1)
    y_shifted = np.roll(y, lag)
    _, Cxy_null = coherence(x, y_shifted, fs=fs, nperseg=nperseg, nw=3)
    null_coherences[i] = Cxy_null
```
</details>

<details>
<summary><b>Hint 3 — How many iterations are enough?</b></summary>

200 iterations is sufficient for a stable 95th percentile.  More iterations
(e.g., 1000) give a smoother null but take longer.  For this exercise,
200 is a good balance.  Print progress every 50 iterations.

```python
for i in range(n_null):
    if i % 50 == 0:
        print(f'  null iteration {i}/{n_null} ...')
    lag = rng.integers(int(1 * fs), len(y) - 1)
    y_shifted = np.roll(y, lag)
    _, Cxy_null = coherence(x, y_shifted, fs=fs, nperseg=nperseg, nw=3)
    null_coherences[i] = Cxy_null

null_95 = np.percentile(null_coherences, 95, axis=0)
```
</details>

In [ ]:
# Checkpoint 3
assert null_coherences.shape[0] == n_null, "Wrong number of null iterations"
assert null_coherences.shape[1] == len(freqs_coh), "Null coherence shape mismatch"
assert np.all(null_95 >= 0) and np.all(null_95 <= 1), "Null percentile out of range"
print(f"Checkpoint 3 passed — null_coherences shape: {null_coherences.shape}, "
      f"95th pctl range: [{null_95.min():.4f}, {null_95.max():.4f}]")

Now plot the observed coherence against the null threshold:

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(freqs_coh, Cxy, linewidth=1.0, color='C2', label='Observed')
ax.plot(freqs_coh, null_95, linewidth=1.0, color='gray', linestyle='--',
        label='95th pctl null')
ax.fill_between(freqs_coh, 0, null_95, alpha=0.1, color='gray')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Coherence')
ax.set_xlim(0, 200)
ax.set_ylim(0, 1)
ax.set_title(f'Coherence vs Null — Ch {ch_x} vs Ch {ch_y}')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Section 4 — Interpreting Frequency-Specific Coupling

Identify the frequency bands where observed coherence exceeds the null.
Summarize the coupling strength in theta (6–10 Hz) and gamma (30–90 Hz).

> **Think about it:** Would nearby channels (e.g., 0 vs 1) show higher
> coherence than distant ones (e.g., 0 vs 15)?  How does this relate
> to the spatial scale of different oscillations?

In [ ]:
# ──────────────────────────────────────────────────────
# YOUR TURN: Summarize coherence in frequency bands
# ──────────────────────────────────────────────────────
# 1. Create boolean masks for theta (6–10 Hz) and gamma (30–90 Hz)
# 2. Compute mean coherence in each band
# 3. Compute the fraction of frequency bins where Cxy > null_95
# 4. Print a summary
# ──────────────────────────────────────────────────────

raise NotImplementedError("Summarize frequency-band coherence")

<details>
<summary><b>Hint 1 — Conceptual</b></summary>

A boolean mask selects frequency bins: `theta_mask = (freqs >= 6) & (freqs <= 10)`.
Use `np.mean(Cxy[mask])` for the average coherence in that band.
Compare `Cxy > null_95` element-wise to find significant bins.
</details>

<details>
<summary><b>Hint 2 — API</b></summary>

```python
theta_mask = (freqs_coh >= 6) & (freqs_coh <= 10)
gamma_mask = (freqs_coh >= 30) & (freqs_coh <= 90)
significant = Cxy > null_95
```
</details>

<details>
<summary><b>Hint 3 — Partial code</b></summary>

```python
significant = Cxy > null_95
theta_mask = (freqs_coh >= 6) & (freqs_coh <= 10)
gamma_mask = (freqs_coh >= 30) & (freqs_coh <= 90)

print(f'Theta (6-10 Hz): mean coh = {np.mean(Cxy[theta_mask]):.3f}, '
      f'{np.mean(significant[theta_mask])*100:.0f}% significant')
print(f'Gamma (30-90 Hz): mean coh = {np.mean(Cxy[gamma_mask]):.3f}, '
      f'{np.mean(significant[gamma_mask])*100:.0f}% significant')
```
</details>

In [ ]:
# Checkpoint 4
assert 0 <= np.mean(Cxy[theta_mask]) <= 1, "Theta coherence out of range"
print(f"Checkpoint 4 passed — theta coherence = {np.mean(Cxy[theta_mask]):.3f}, "
      f"gamma coherence = {np.mean(Cxy[gamma_mask]):.3f}")

---
## Reflection

1. **Channel distance:** Try computing coherence for a nearby pair (ch 0 vs 1)
   and a distant pair (ch 0 vs 15).  How does the coherence profile change?
   Which frequency bands are most affected by distance?

2. **Segment length:** You used 2-second segments.  What happens if you use
   0.5 s or 5 s?  How does this affect the coherence estimate and the null
   distribution floor?

3. **Beyond pairwise:** For a full characterization of network coupling,
   you'd compute coherence for all channel pairs.  The `lib/mtcsd.py` module
   computes the full multitaper cross-spectral density matrix efficiently.
   The SS25 Ex10 notebook (`Exercise_09.ipynb`) demonstrates this with 96
   channels — see `sourcedata/moodle/ss25-public/.../Exercise_10/` if interested.